# Import Dataset Daily V2

Notebook ini mengambil data **daily** dari Open-Meteo Archive API dengan fitur harian lengkap (tanpa data hourly).

### Daily Features (21 total)
- Temperature: `temperature_2m_max`, `temperature_2m_min`, `temperature_2m_mean`
- Humidity: `relative_humidity_2m_mean`, `relative_humidity_2m_max`, `relative_humidity_2m_min`
- Pressure (MSL): `pressure_msl_mean`, `pressure_msl_max`, `pressure_msl_min`
- Surface Pressure: `surface_pressure_mean`, `surface_pressure_max`, `surface_pressure_min`
- Wind Speed: `wind_speed_10m_mean`, `wind_speed_10m_max`, `wind_speed_10m_min`
- Wind Gusts: `wind_gusts_10m_mean`, `wind_gusts_10m_max`, `wind_gusts_10m_min`
- Other: `weather_code`, `rain_sum`, `winddirection_10m_dominant`


In [13]:
import requests
import pandas as pd
from datetime import datetime
import time
import os

# Coordinates and Parameters
LATITUDE = -7.0520702239386175
LONGITUDE = 110.43532807750137
TIMEZONE = "Asia/Jakarta"
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Rate Limiting Config
MAX_RETRIES = 5
BASE_DELAY = 2  # Base delay between requests (seconds)
RETRY_DELAY = 30  # Initial retry delay for 429 errors (seconds)

# Daily feature list only (no hourly)
DAILY_FEATURES = [
    "temperature_2m_max", "temperature_2m_min", "weather_code",
    "relative_humidity_2m_mean", "pressure_msl_mean", "wind_speed_10m_mean",
    "temperature_2m_mean", "relative_humidity_2m_max", "relative_humidity_2m_min",
    "pressure_msl_max", "pressure_msl_min", "wind_speed_10m_max",
    "rain_sum", "surface_pressure_mean", "surface_pressure_max", "surface_pressure_min",
    "winddirection_10m_dominant", "wind_gusts_10m_min", "wind_speed_10m_min",
    "wind_gusts_10m_max", "wind_gusts_10m_mean"
]

print(f"Daily features: {len(DAILY_FEATURES)}")


Daily features: 21


In [14]:
# Weather Condition Mapping (Optional helper)
def map_weather_code(code):
    '''Maps WMO weather code to a simple condition string.'''
    if code is None:
        return "Unknown"
    if code == 0:
        return "Clear"
    elif code in [1, 2]:
        return "Partially cloudy"
    elif code in [3, 45, 48]:
        return "Overcast"
    elif code in [51, 53, 55]:
        return "Rain"
    elif code in [61, 63, 65]:
        return "Rain, Overcast"
    elif code in [80, 81, 82]:
        return "Rain, Partially cloudy"
    elif code in [95, 96, 99]:
        return "Rain"
    else:
        return "Unknown"


In [15]:
def fetch_daily_data_chunk(start_date, end_date, retries=0):
    '''Fetch daily data for a specific date range with retry logic.'''
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "daily": DAILY_FEATURES,
        "timezone": TIMEZONE,
    }

    try:
        response = requests.get(API_URL, params=params)

        # Handle rate limiting (429)
        if response.status_code == 429:
            if retries < MAX_RETRIES:
                wait_time = RETRY_DELAY * (2 ** retries)  # Exponential backoff
                print(f"   ⚠️ Rate limited! Waiting {wait_time}s before retry {retries + 1}/{MAX_RETRIES}...")
                time.sleep(wait_time)
                return fetch_daily_data_chunk(start_date, end_date, retries + 1)
            print(f"   ❌ Max retries reached for {start_date} to {end_date}")
            return None

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        if retries < MAX_RETRIES:
            wait_time = RETRY_DELAY * (2 ** retries)
            print(f"   ⚠️ Error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)
            return fetch_daily_data_chunk(start_date, end_date, retries + 1)
        print(f"   ❌ Failed after {MAX_RETRIES} retries: {e}")
        return None


In [16]:
def fetch_historical_daily_data():
    '''Fetch daily data from 2000 to today in yearly chunks.'''
    today = datetime.now()
    start_year = 2000
    end_year = today.year

    all_data = []
    total_years = end_year - start_year + 1

    for idx, year in enumerate(range(start_year, end_year + 1)):
        start_date = f"{year}-01-01"
        end_date = today.strftime("%Y-%m-%d") if year == end_year else f"{year}-12-31"

        print(f"[{idx + 1}/{total_years}] Fetching data for {year} ({start_date} to {end_date})...")

        data = fetch_daily_data_chunk(start_date, end_date)
        if data is None:
            print(f"   ⏭️ Skipping year {year} due to errors")
            continue

        daily_data = data.get("daily", {})
        if not daily_data:
            print(f"   ⚠️ No daily data found for {year}.")
            continue

        df_daily = pd.DataFrame({
            "date": daily_data.get("time"),
            # Temperature
            "temp_max": daily_data.get("temperature_2m_max"),
            "temp_min": daily_data.get("temperature_2m_min"),
            "temp_mean": daily_data.get("temperature_2m_mean"),
            # Weather & rain
            "weather_code": daily_data.get("weather_code"),
            "rain_sum": daily_data.get("rain_sum"),
            # Humidity
            "humidity_mean": daily_data.get("relative_humidity_2m_mean"),
            "humidity_max": daily_data.get("relative_humidity_2m_max"),
            "humidity_min": daily_data.get("relative_humidity_2m_min"),
            # MSL Pressure
            "pressure_msl_mean": daily_data.get("pressure_msl_mean"),
            "pressure_msl_max": daily_data.get("pressure_msl_max"),
            "pressure_msl_min": daily_data.get("pressure_msl_min"),
            # Surface Pressure
            "surface_pressure_mean": daily_data.get("surface_pressure_mean"),
            "surface_pressure_max": daily_data.get("surface_pressure_max"),
            "surface_pressure_min": daily_data.get("surface_pressure_min"),
            # Wind Speed
            "windspeed_mean": daily_data.get("wind_speed_10m_mean"),
            "windspeed_max": daily_data.get("wind_speed_10m_max"),
            "windspeed_min": daily_data.get("wind_speed_10m_min"),
            # Wind Gusts
            "wind_gusts_mean": daily_data.get("wind_gusts_10m_mean"),
            "wind_gusts_max": daily_data.get("wind_gusts_10m_max"),
            "wind_gusts_min": daily_data.get("wind_gusts_10m_min"),
            # Wind Direction
            "wind_direction_dominant": daily_data.get("winddirection_10m_dominant"),
        })

        df_daily["date"] = pd.to_datetime(df_daily["date"])
        df_daily["year"] = df_daily["date"].dt.year
        df_daily["month"] = df_daily["date"].dt.month
        df_daily["day"] = df_daily["date"].dt.day
        df_daily["conditions"] = df_daily["weather_code"].apply(map_weather_code)

        all_data.append(df_daily)
        print(f"   ✅ {len(df_daily):,} records fetched")

        if idx < total_years - 1:  # Don't wait after last request
            print(f"   ⏳ Waiting {BASE_DELAY}s before next request...")
            time.sleep(BASE_DELAY)

    if not all_data:
        print("❌ No data fetched.")
        return None

    df = pd.concat(all_data, ignore_index=True)
    df["id"] = range(len(df))

    # Organize columns
    output_columns = [
        "id", "date", "year", "month", "day",
        "temp_max", "temp_min", "temp_mean",
        "weather_code", "conditions", "rain_sum",
        "humidity_mean", "humidity_max", "humidity_min",
        "pressure_msl_mean", "pressure_msl_max", "pressure_msl_min",
        "surface_pressure_mean", "surface_pressure_max", "surface_pressure_min",
        "windspeed_mean", "windspeed_max", "windspeed_min",
        "wind_gusts_mean", "wind_gusts_max", "wind_gusts_min",
        "wind_direction_dominant",
    ]

    return df[output_columns]


In [ ]:
# Execute the data fetch
df = fetch_historical_daily_data()

if df is not None:
    # Ensure output directory exists
    output_dir = "../data"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    output_file = os.path.join(output_dir, "historical_data_daily.csv")
    df.to_csv(output_file, index=False)
    print(f" 🎉 Data successfully saved to {output_file}")
    print(f"📊 Total records: {len(df):,}")
    print(f"📋 Total columns: {len(df.columns)}")


[1/26] Fetching data for 2000 (2000-01-01 to 2000-12-31)...
   ✅ 366 records fetched
   ⏳ Waiting 2s before next request...
[2/26] Fetching data for 2001 (2001-01-01 to 2001-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[3/26] Fetching data for 2002 (2002-01-01 to 2002-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[4/26] Fetching data for 2003 (2003-01-01 to 2003-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[5/26] Fetching data for 2004 (2004-01-01 to 2004-12-31)...
   ✅ 366 records fetched
   ⏳ Waiting 2s before next request...
[6/26] Fetching data for 2005 (2005-01-01 to 2005-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[7/26] Fetching data for 2006 (2006-01-01 to 2006-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[8/26] Fetching data for 2007 (2007-01-01 to 2007-12-31)...
   ✅ 365 records fetched
   ⏳ Waiting 2s before next request...
[9/26] F

In [ ]:
# Preview the data
if df is not None:
    print("=== Column List ===")
    for i, col in enumerate(df.columns):
        print(f"{i+1:2}. {col}")

    print("=== Data Preview (First 5 rows) ===")
    display(df.head())

    print("=== Data Info ===")
    df.info()


=== Column List ===
 1. id
 2. date
 3. year
 4. month
 5. day
 6. temp_max
 7. temp_min
 8. temp_mean
 9. weather_code
10. conditions
11. rain_sum
12. humidity_mean
13. humidity_max
14. humidity_min
15. pressure_msl_mean
16. pressure_msl_max
17. pressure_msl_min
18. surface_pressure_mean
19. surface_pressure_max
20. surface_pressure_min
21. windspeed_mean
22. windspeed_max
23. windspeed_min
24. wind_gusts_mean
25. wind_gusts_max
26. wind_gusts_min
27. wind_direction_dominant
=== Data Preview (First 5 rows) ===


,id,date,year,month,day,temp_max,temp_min,temp_mean,weather_code,conditions,...,surface_pressure_mean,surface_pressure_max,surface_pressure_min,windspeed_mean,windspeed_max,windspeed_min,wind_gusts_mean,wind_gusts_max,wind_gusts_min,wind_direction_dominant
0,0,2000-01-01,2000,1,1,27.5,20.8,24.1,53,Rain,...,983.6,985.4,981.3,6.3,12.1,1.4,17.3,30.2,6.8,300
1,1,2000-01-02,2000,1,2,27.0,21.6,23.8,61,"Rain, Overcast",...,983.5,984.9,981.8,7.5,13.0,1.0,19.9,37.1,4.7,310
2,2,2000-01-03,2000,1,3,24.9,22.1,23.3,61,"Rain, Overcast",...,984.1,985.5,982.1,5.2,8.0,2.9,14.6,21.2,6.1,317
3,3,2000-01-04,2000,1,4,27.6,21.9,24.3,53,Rain,...,984.6,986.5,982.4,3.6,6.1,1.3,11.4,21.2,4.3,298
4,4,2000-01-05,2000,1,5,28.9,21.3,24.2,65,"Rain, Overcast",...,985.3,987.3,982.9,3.8,5.6,1.8,14.6,26.6,7.2,132


=== Data Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9477 entries, 0 to 9476
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       9477 non-null   int64         
 1   date                     9477 non-null   datetime64[ns]
 2   year                     9477 non-null   int32         
 3   month                    9477 non-null   int32         
 4   day                      9477 non-null   int32         
 5   temp_max                 9477 non-null   float64       
 6   temp_min                 9477 non-null   float64       
 7   temp_mean                9477 non-null   float64       
 8   weather_code             9477 non-null   int64         
 9   conditions               9477 non-null   object        
 10  rain_sum                 9477 non-null   float64       
 11  humidity_mean            9477 non-null   int64         
 12  humidity_max    

In [ ]:
# Summary Statistics
if df is not None:
    print("=== Summary Statistics ===")
    display(df.describe())

    print("=== Missing Values ===")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_pct
    })
    display(missing_df[missing_df["Missing Count"] > 0])


=== Summary Statistics ===


,id,date,year,month,day,temp_max,temp_min,temp_mean,weather_code,rain_sum,...,surface_pressure_mean,surface_pressure_max,surface_pressure_min,windspeed_mean,windspeed_max,windspeed_min,wind_gusts_mean,wind_gusts_max,wind_gusts_min,wind_direction_dominant
count,9477.000000,9477,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,...,9477.00000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000,9477.000000
mean,4738.000000,2012-12-21 00:00:00,2012.473251,6.511132,15.718160,29.925367,22.183824,25.528511,46.273399,6.509381,...,986.39561,988.208969,984.307545,6.159196,11.300791,1.955292,16.847779,29.354279,6.811934,193.082410
min,0.000000,2000-01-01 00:00:00,2000.000000,1.000000,1.000000,23.300000,17.000000,22.400000,0.000000,0.000000,...,980.90000,982.100000,978.700000,1.700000,3.700000,0.000000,8.500000,11.500000,0.700000,0.000000
25%,2369.000000,2006-06-27 00:00:00,2006.000000,4.000000,8.000000,28.300000,21.500000,24.500000,51.000000,0.200000,...,985.50000,987.200000,983.500000,4.300000,8.400000,0.800000,13.500000,24.100000,4.700000,130.000000
50%,4738.000000,2012-12-21 00:00:00,2012.000000,7.000000,16.000000,29.500000,22.200000,25.300000,55.000000,3.000000,...,986.50000,988.300000,984.400000,5.500000,10.400000,1.500000,15.700000,27.700000,5.800000,158.000000
75%,7107.000000,2019-06-17 00:00:00,2019.000000,10.000000,23.000000,31.500000,23.000000,26.500000,63.000000,10.000000,...,987.30000,989.200000,985.200000,7.200000,13.300000,2.400000,18.800000,32.800000,7.600000,292.000000
max,9476.000000,2025-12-11 00:00:00,2025.000000,12.000000,31.000000,37.800000,25.400000,30.400000,65.000000,171.500000,...,990.90000,993.000000,989.400000,26.900000,34.700000,22.900000,58.500000,73.800000,46.400000,360.000000
std,2735.918584,NaN,7.486660,3.443386,8.802151,2.317263,1.165909,1.344250,23.498863,9.066065,...,1.36812,1.470321,1.332899,2.657051,4.187879,1.935547,4.961694,7.464940,3.880073,85.328226


=== Missing Values ===


,Missing Count,Missing %
